# TESSERA Master Notebook v4
**Objective**: Orchestrate the TESSERA preprocessing and inference pipeline.

This notebook calls functions defined in `V4_TESSERA_Worker.ipynb` to perform environment setup and per-tile processing.

In [1]:
import nbformat
from IPython import get_ipython
from pathlib import Path
import sys, os

def run_notebook_inplace(path, clear_functions=True):
    """Execute all code cells from a notebook in the current runtime."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Notebook not found at: {path}")

    nb = nbformat.read(open(path), as_version=4)
    shell = get_ipython()

    if clear_functions:
        # Parse all function/class names defined in the notebook and delete them
        import ast
        for cell in nb.cells:
            if cell.cell_type != "code":
                continue
            try:
                tree = ast.parse(cell.source)
                for node in ast.walk(tree):
                    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                        if node.name in shell.user_ns:
                            del shell.user_ns[node.name]
            except SyntaxError:
                pass  # skip unparseable cells

    for cell in nb.cells:
        if cell.cell_type == "code":
            shell.run_cell(cell.source)

    print(f"✅ Executed notebook in current kernel: {path}")

# ── Path Logic for Link ──
# Flag: switch between production (shared drive) and local testing
#run_production = False  # <-- Use for private or local testing
#run_production = True   # <-- Use for production deployment

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    # if run_production:
        # Placeholder for production path structure if different
    #worker_dir = Path('/content/drive/MyDrive/TESSERA_PROD/REPO/local_work/notebooks-test/')
    # else:
    worker_dir = Path('/content/drive/MyDrive/Colab Notebooks/TESSERA/')
else:
    # Local system
    worker_dir = Path('./')

worker_file = "V4_TESSERA_Worker.ipynb"
WORKER_PATH = worker_dir / worker_file

# Link to the worker now:
run_notebook_inplace(WORKER_PATH)

Mounted at /content/drive
✅ Executed notebook in current kernel: /content/drive/MyDrive/Colab Notebooks/TESSERA/V4_TESSERA_Worker.ipynb


### Step 1 — Runtime check

In [2]:
check_runtime()


========== System ==========
Python   : 3.12.12
OS       : Linux 6.6.113+

========== GPU ==========
GPU      : NVIDIA L4, 23034 MiB
Wed Mar 18 16:56:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|     

### Step 2 — Install dependencies

In [3]:
install_dependencies()

import numpy as np
import rasterio
import pyproj
import rioxarray, xarray, stackstac, dask
import subprocess, time, threading
import pystac_client, planetary_computer

from rasterio.transform import from_bounds
from rasterio.crs import CRS
from pyproj import Transformer
from shapely.geometry import box

print(f"rasterio   : {rasterio.__version__}")
print(f"xarray     : {xarray.__version__}")
print(f"stackstac  : {stackstac.__version__}")
print(f"dask       : {dask.__version__}")
print(f"pyproj     : {pyproj.__version__}")
print("All core imports OK.")

Checking core dependencies...
Installing: affine, dask[distributed], fiona, pyproj, rasterio, rioxarray, shapely, stackstac, xarray, pystac-client, pystac, planetary-computer, gdown, tqdm...
Installation completed successfully.
rasterio   : 1.5.0
xarray     : 2026.2.0
stackstac  : 0.5.1
dask       : 2026.1.1
pyproj     : 3.7.2
All core imports OK.


### Step 3 — Mount Drive & set paths

In [4]:
# Configures the base paths for data and repo
global_paths = setup_paths(in_colab=IN_COLAB)

Paths configured:
  Drive base  : /content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline
  Repo        : /content/drive/MyDrive/Colab Notebooks/TESSERA/REPO
  Data        : /content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline/data
  Temp        : /content/tmp_tessera
  Checkpoint  : /content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline/checkpoints
  Output      : /content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline/output


### Step 4 — Clone TESSERA repository

In [5]:
# Clone logic from v3 with FORCE_RECLONE enabled
clone_tessera_repo(repo_dir=global_paths.REPO_DIR,
                   FORCE_RECLONE=False,
                   BRANCH='alpha_version_1.0')

Repo already present at: /content/drive/MyDrive/Colab Notebooks/TESSERA/REPO
Active branch: alpha_version_1.0
REPO_DIR in sys.path: True


### Step 5 — Apply SCL patch (thin cirrus)

In [6]:
# Apply the cloud masking patch defined in the worker
apply_scl_patch(repo_dir=global_paths.REPO_DIR)

SCL_INVALID before patch: {0, 1, 2, 3, 8, 9, np.nan, 10}
SCL patch already applied (class 10 present).
Verified: SCL_INVALID = {0, 1, 2, 3, 8, 9, np.nan, 10}


### Step 6 — Set executable permissions on binaries & scripts

In [7]:
# Set permissions for shell scripts and Rust binaries
# ── Call it ───────────────────────────────────────────────────────────
set_permissions(repo_dir=global_paths.REPO_DIR)

Setting executable permissions:
  ✓  tessera_preprocessing/s1_s2_downloader.sh
  ✓  tessera_preprocessing/s1_s2_stacker.sh
  ✓  tessera_infer/infer_all_tiles.sh
  ✓  tessera_preprocessing/s1_stack
  ✓  tessera_preprocessing/s2_stack


### Step 7 — Inference Setup


In [8]:
# Localize checkpoint and patch launch scripts
# ── Call it ───────────────────────────────────────────────────────────
# using: ckpt_filename: str = 'best_model_fsdp_20250427_084307.pt')
global_paths.CKPT_LOCAL = setup_inference(repo_dir=global_paths.REPO_DIR,
                                          checkpoint_dir=global_paths.CHECKPOINT,
                                          in_colab=IN_COLAB)

Searching Drive for 'best_model_fsdp_20250427_084307.pt'...
  Found: /content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline/checkpoints/best_model_fsdp_20250427_084307.pt
  Copying to /content/ for faster load (no spaces)...
  Done — 7163 MB
PYTHON_ENV already patched.


## Pipeline Configuration

In [9]:
# Link to the worker to collect any last minute changes:
run_notebook_inplace(WORKER_PATH)

✅ Executed notebook in current kernel: /content/drive/MyDrive/Colab Notebooks/TESSERA/V4_TESSERA_Worker.ipynb


In [10]:
AREAS = {
    "Soubre_Test_Tile": {
        "lon": -6.60,
        "lat": 5.78,
        "size_m": 5000,
        "res": 10.0,
        "epsg": 32630
    }
}

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

In [ ]:
# Running the TESSERA Pipeline
for i, (area_name, params) in enumerate(AREAS.items()):
    print(f"\n=== Area: {area_name} ===")
    # Step 8: Generate ROI for this area
    roi_path = worker_step_roi(area_name, params, global_paths)

    for j, year in enumerate(YEARS):
        print(f"--- Processing {year} ---")
        # Step 9-15: Execute worker for this specific period
        verbose = (i == 0 and j == 0)
        run_worker_pipeline(area_name, params, year, roi_path, global_paths,
                            verbose)


=== Area: Soubre_Test_Tile ===
ROI Generated: /content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline/data/Soubre_Test_Tile_roi.tif
--- Processing 2020 ---

[WORKER] Pipeline execution start: Soubre_Test_Tile (2020)

[S2] Starting download for 2020...
[S2] Done in 1.2 min

[S1] Starting download for 2020...
[S1] Done in 0.9 min

[Stacking] Launching s1_stack and s2_stack in parallel...
Both processes finished.

  ✓ s1_stack  (rc=0, 0.8s)
  ✓ s2_stack  (rc=0, 1.9s)

[Retiling] Slicing stacks into 40x40 patches...
  ✓  rc=0  (2.4s)

[Inference] Running TESSERA model...
  Checkpoint : /content/checkpoint.pt
  Tiles dir  : /content/tmp_tessera/retiled_d_pixel
  Output dir : /content/tmp_tessera/representation_retiled

  ⏳ Long process — estimated 10 minutes, no output until complete...

  ✓  rc=0  (6.1 min)
h 1/2 - 1024 samples - 1.2s elapsed - 874.5 samples/sec - GPU 0 Memory: 12371.9MB allocated, 13768.0MB reserved, 13492.9MB peak, 22563.1MB total
2026-03-18 17:08:25 - I